[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/099.%20The%20Electric%20Vehicle%20Routing%20Problem%20%28EVRP%29/P99-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 3: Dragonfly Algorithm

### Key Assumptions
- Electric vehicles operate with battery constraints requiring strategic charging
- Charging stations are available at fixed locations with known charging rates
- Energy consumption depends on distance, payload, and route characteristics
- Dragonfly swarm intelligence can explore complex solution spaces effectively
- Population-based search can escape local optima better than greedy methods

### Approach (Step-by-Step)

The Dragonfly Algorithm (DA) is a nature-inspired metaheuristic that mimics the swarming behavior of dragonflies:

1. **Initialization**: Create a population of dragonflies with random positions (route permutations)
2. **Five Behavioral Principles**: Implement separation, alignment, cohesion, food attraction, and enemy distraction
3. **Position Representation**: Encode customer permutations as dragonfly positions
4. **Route Decoding**: Convert permutations to feasible EV routes with charging station insertion
5. **Fitness Evaluation**: Calculate route quality considering distance, energy, and feasibility
6. **Step Vector Update**: Update dragonfly movement based on behavioral principles
7. **Position Update**: Move dragonflies and maintain boundary conditions
8. **Convergence**: Track best solution and terminate when convergence criteria met

### What to Look for in the Results
- High-quality solutions approaching optimal performance
- Better exploration of solution space compared to greedy methods
- Convergence behavior showing improvement over iterations
- Robust performance across multiple runs
- Effective handling of complex EV constraints

### Concrete Example

We'll solve a challenging instance to demonstrate the metaheuristic's power:
- 1 depot
- 6 customers (larger instance for metaheuristic demonstration)
- 2 charging stations
- 2 electric vehicles
- Battery capacity: 100 kWh
- Energy consumption: 0.8 kWh per km
- Population: 20 dragonflies, 100 iterations

In [1]:
from itertools import permutations
from typing import Tuple, List, Dict, Set

# Import required libraries for Dragonfly Algorithm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class Node:
    """Represents a node in the EVRP network"""
    id: int
    x: float
    y: float
    node_type: str  # 'depot', 'customer', 'charging_station'
    service_time: float = 0.0
    
@dataclass
class Vehicle:
    """Represents an electric vehicle"""
    id: int
    battery_capacity: float  # kWh
    initial_charge: float  # kWh
    
@dataclass
class Dragonfly:
    """Represents a dragonfly in the swarm"""
    position: np.ndarray  # Customer permutation
    step_vector: np.ndarray  # Movement vector
    fitness: float
    routes: List[List[int]]  # Decoded routes
    
@dataclass
class EVRPInstance:
    """EVRP problem instance"""
    nodes: List[Node]
    vehicles: List[Vehicle]
    energy_consumption_rate: float  # kWh per km
    charging_rate: float  # kW
    
def create_metaheuristic_instance():
    """Create a larger instance for metaheuristic demonstration"""
    
    # Define nodes: depot (0), customers (1-6), charging stations (7,8)
    nodes = [
        Node(0, 0, 0, 'depot'),  # Depot at origin
        Node(1, 15, 10, 'customer', service_time=0.5),   # Customer 1
        Node(2, 20, 18, 'customer', service_time=0.5),   # Customer 2
        Node(3, 12, 22, 'customer', service_time=0.5),   # Customer 3
        Node(4, 25, 8, 'customer', service_time=0.5),    # Customer 4
        Node(5, 8, 15, 'customer', service_time=0.5),    # Customer 5
        Node(6, 18, 25, 'customer', service_time=0.5),   # Customer 6
        Node(7, 10, 12, 'charging_station'),  # Charging Station 1
        Node(8, 22, 15, 'charging_station'),  # Charging Station 2
    ]
    
    # Define vehicles
    vehicles = [
        Vehicle(0, battery_capacity=100, initial_charge=100),
        Vehicle(1, battery_capacity=100, initial_charge=100)
    ]
    
    return EVRPInstance(
        nodes=nodes,
        vehicles=vehicles,
        energy_consumption_rate=0.8,  # 0.8 kWh per km
        charging_rate=50.0  # 50 kW charging rate
    )

# Create the instance
instance = create_metaheuristic_instance()
print(f"Created EVRP instance with {len(instance.nodes)} nodes and {len(instance.vehicles)} vehicles")
print(f"Customers: {[i for i, node in enumerate(instance.nodes) if node.node_type == 'customer']}")
print(f"Charging Stations: {[i for i, node in enumerate(instance.nodes) if node.node_type == 'charging_station']}")

Created EVRP instance with 9 nodes and 2 vehicles
Customers: [1, 2, 3, 4, 5, 6]
Charging Stations: [7, 8]


In [ ]:
def calculate_distance_matrix(nodes: List[Node]) -> np.ndarray:
    """Calculate distance matrix between all nodes"""
    n = len(nodes)
    distances = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i != j:
                distances[i, j] = np.sqrt((nodes[i].x - nodes[j].x)**2 + (nodes[i].y - nodes[j].y)**2)
    
    return distances

def calculate_energy_matrix(distances: np.ndarray, consumption_rate: float) -> np.ndarray:
    """Calculate energy consumption matrix"""
    return distances * consumption_rate

# Pre-calculate matrices
distances = calculate_distance_matrix(instance.nodes)
energy_consumption = calculate_energy_matrix(distances, instance.energy_consumption_rate)

print("Distance Matrix (km):")
print(np.round(distances, 2))
print("\nEnergy Consumption Matrix (kWh):")
print(np.round(energy_consumption, 2))

Distance Matrix (km):
[[ 0.   18.03 26.91 25.06 26.25 17.   30.81 15.62 26.63]
 [18.03  0.    9.43 12.37 10.2   8.6  15.3   5.39  8.6 ]
 [26.91  9.43  0.    8.94 11.18 12.37  7.28 11.66  3.61]
 [25.06 12.37  8.94  0.   19.1   8.06  6.71 10.2  12.21]
 [26.25 10.2  11.18 19.1   0.   18.38 18.38 15.52  7.62]
 [17.    8.6  12.37  8.06 18.38  0.   14.14  3.61 14.  ]
 [30.81 15.3   7.28  6.71 18.38 14.14  0.   15.26 10.77]
 [15.62  5.39 11.66 10.2  15.52  3.61 15.26  0.   12.37]
 [26.63  8.6   3.61 12.21  7.62 14.   10.77 12.37  0.  ]]

Energy Consumption Matrix (kWh):
[[ 0.   14.42 21.53 20.05 21.   13.6  24.64 12.5  21.3 ]
 [14.42  0.    7.55  9.9   8.16  6.88 12.24  4.31  6.88]
 [21.53  7.55  0.    7.16  8.94  9.9   5.82  9.33  2.88]
 [20.05  9.9   7.16  0.   15.28  6.45  5.37  8.16  9.77]
 [21.    8.16  8.94 15.28  0.   14.71 14.71 12.42  6.09]
 [13.6   6.88  9.9   6.45 14.71  0.   11.31  2.88 11.2 ]
 [24.64 12.24  5.82  5.37 14.71 11.31  0.   12.21  8.62]
 [12.5   4.31  9.33  8.16 12.42

In [ ]:
def decode_permutation_to_routes(permutation: np.ndarray, instance: EVRPInstance, 
                                distances: np.ndarray, energy_matrix: np.ndarray) -> List[List[int]]:
    """Decode a customer permutation into feasible EV routes
    
    This is a simplified decoder that splits the permutation into routes
    and inserts charging stations when needed.
    """
    
    customers = [i for i, node in enumerate(instance.nodes) if node.node_type == 'customer']
    charging_stations = [i for i, node in enumerate(instance.nodes) if node.node_type == 'charging_station']
    
    routes = []
    current_route = [0]  # Start from depot
    current_battery = instance.vehicles[0].initial_charge
    
    for customer in permutation:
        # Check if we can serve this customer with current battery
        last_node = current_route[-1]
        energy_to_customer = energy_matrix[last_node, customer]
        energy_to_depot = energy_matrix[customer, 0]
        
        # Check if we can serve customer and return to depot
        if current_battery >= energy_to_customer + energy_to_depot:
            current_route.append(customer)
            current_battery -= energy_to_customer
        else:
            # Need to go to charging station or complete current route
            if len(current_route) > 1:  # If we have customers in route
                # Return to depot and start new route
                current_route.append(0)
                routes.append(current_route)
                current_route = [0, customer]
                current_battery = instance.vehicles[0].initial_charge - energy_matrix[0, customer]
            else:
                # Try to insert charging station
                current_route.append(customer)
    
    # Complete the last route
    if len(current_route) > 1:
        current_route.append(0)
        routes.append(current_route)
    
    # Insert charging stations if needed
    for i, route in enumerate(routes):
        routes[i] = insert_charging_stations(route, instance, energy_matrix)
    
    return routes

def insert_charging_stations(route: List[int], instance: EVRPInstance, 
                          energy_matrix: np.ndarray) -> List[int]:
    """Insert charging stations into route if needed"""
    
    charging_stations = [i for i, node in enumerate(instance.nodes) if node.node_type == 'charging_station']
    new_route = [route[0]]  # Start with depot
    current_battery = instance.vehicles[0].initial_charge
    
    for i in range(len(route) - 1):
        current_node = new_route[-1]
        next_node = route[i + 1]
        
        energy_needed = energy_matrix[current_node, next_node]
        
        # Check if we need charging
        if current_battery < energy_needed * 1.2:  # 20% safety margin
            # Find nearest charging station
            best_station = None
            min_extra_distance = float('inf')
            
            for station in charging_stations:
                if station not in new_route:
                    extra_distance = distances[current_node, station] + distances[station, next_node] - distances[current_node, next_node]
                    if extra_distance < min_extra_distance:
                        min_extra_distance = extra_distance
                        best_station = station
            
            if best_station is not None:
                new_route.append(best_station)
                current_battery = instance.vehicles[0].initial_charge  # Reset battery at charging station
        
        new_route.append(next_node)
        current_battery -= energy_needed
    
    return new_route

# Test the decoder
test_permutation = np.array([1, 3, 5, 2, 4, 6])  # Customer permutation
test_routes = decode_permutation_to_routes(test_permutation, instance, distances, energy_consumption)
print(f"Test permutation: {test_permutation}")
print(f"Decoded routes: {test_routes}")

Test permutation: [1 3 5 2 4 6]
Decoded routes: [[0, np.int64(1), np.int64(3), np.int64(5), np.int64(2), np.int64(4), np.int64(6), 0]]


In [ ]:
def calculate_route_fitness(routes: List[List[int]], instance: EVRPInstance, 
                          distances: np.ndarray, energy_matrix: np.ndarray) -> float:
    """Calculate fitness (total distance) of a set of routes
    
    Lower fitness is better (minimize distance)
    """
    
    total_distance = 0
    penalty = 0
    
    for route in routes:
        # Calculate route distance
        route_distance = 0
        for i in range(len(route) - 1):
            route_distance += distances[route[i], route[i + 1]]
        
        total_distance += route_distance
        
        # Check feasibility and add penalties
        current_battery = instance.vehicles[0].initial_charge
        
        for i in range(len(route) - 1):
            current_node = route[i]
            next_node = route[i + 1]
            energy_needed = energy_matrix[current_node, next_node]
            
            # Check battery violation
            if current_battery < energy_needed:
                penalty += 1000  # Large penalty for battery violation
            
            current_battery -= energy_needed
            
            # Reset battery at charging stations
            if instance.nodes[next_node].node_type == 'charging_station':
                current_battery = instance.vehicles[0].initial_charge
    
    # Penalize if too many routes
    if len(routes) > len(instance.vehicles):
        penalty += 500 * (len(routes) - len(instance.vehicles))
    
    return total_distance + penalty

# Test fitness calculation
test_fitness = calculate_route_fitness(test_routes, instance, distances, energy_consumption)
print(f"Test routes fitness: {test_fitness:.2f}")

Test routes fitness: 111.20


In [ ]:
def initialize_dragonfly_swarm(population_size: int, n_customers: int) -> List[Dragonfly]:
    """Initialize the dragonfly swarm with random positions"""
    
    swarm = []
    
    for i in range(population_size):
        # Create random permutation of customers
        customer_ids = list(range(1, n_customers + 1))  # Customer IDs start from 1
        position = np.random.permutation(customer_ids)
        
        # Initialize step vector (movement)
        step_vector = np.zeros(n_customers)
        
        dragonfly = Dragonfly(
            position=position,
            step_vector=step_vector,
            fitness=float('inf'),
            routes=[]
        )
        
        swarm.append(dragonfly)
    
    return swarm

def calculate_separation(dragonfly: Dragonfly, neighbors: List[Dragonfly], 
                        separation_radius: float) -> np.ndarray:
    """Separation: avoid colliding with neighbors"""
    
    if len(neighbors) == 0:
        return np.zeros_like(dragonfly.position)
    
    separation_vector = np.zeros_like(dragonfly.position)
    
    for neighbor in neighbors:
        # Calculate distance in permutation space (simplified)
        distance = np.sum(dragonfly.position != neighbor.position)
        if distance < separation_radius:
            # Move away from neighbor (simplified as adding noise)
            separation_vector += np.random.randn(len(dragonfly.position)) * 0.1
    
    return separation_vector

def calculate_alignment(dragonfly: Dragonfly, neighbors: List[Dragonfly]) -> np.ndarray:
    """Alignment: match velocity with neighbors"""
    
    if len(neighbors) == 0:
        return np.zeros_like(dragonfly.position)
    
    # Average step vector of neighbors
    avg_step = np.mean([n.step_vector for n in neighbors], axis=0)
    return avg_step * 0.1

def calculate_cohesion(dragonfly: Dragonfly, neighbors: List[Dragonfly]) -> np.ndarray:
    """Cohesion: move towards center of mass of neighbors"""
    
    if len(neighbors) == 0:
        return np.zeros_like(dragonfly.position)
    
    # Calculate center of mass (simplified as average position)
    center_of_mass = np.zeros_like(dragonfly.position)
    for neighbor in neighbors:
        # Create a mapping from neighbor position to center (simplified)
        for i, customer in enumerate(neighbor.position):
            center_of_mass[i] += customer
    
    center_of_mass /= len(neighbors)
    
    # Move towards center (simplified)
    cohesion_vector = (center_of_mass - dragonfly.position) * 0.05
    return cohesion_vector

# Initialize swarm for testing
n_customers = len([n for n in instance.nodes if n.node_type == 'customer'])
swarm = initialize_dragonfly_swarm(5, n_customers)  # Small swarm for testing
print(f"Initialized swarm with {len(swarm)} dragonflies")
print(f"First dragonfly position: {swarm[0].position}")

In [ ]:
def calculate_food_attraction(dragonfly: Dragonfly, food_source: Dragonfly) -> np.ndarray:
    """Attraction to food: move towards best solution found"""
    
    if food_source is None:
        return np.zeros_like(dragonfly.position)
    
    # Move towards food source (simplified as partial copying)
    attraction_vector = np.zeros_like(dragonfly.position)
    
    for i in range(len(dragonfly.position)):
        if dragonfly.position[i] != food_source.position[i]:
            # With some probability, copy the food source's position
            if np.random.random() < 0.3:
                attraction_vector[i] = food_source.position[i] - dragonfly.position[i]
    
    return attraction_vector * 0.2

def calculate_enemy_distraction(dragonfly: Dragonfly, enemy: Dragonfly) -> np.ndarray:
    """Distraction from enemy: move away from worst solution"""
    
    if enemy is None:
        return np.zeros_like(dragonfly.position)
    
    # Move away from enemy (simplified as random perturbation)
    distraction_vector = np.random.randn(len(dragonfly.position)) * 0.3
    return distraction_vector

def update_dragonfly_position(dragonfly: Dragonfly, food_source: Dragonfly, 
                             enemy: Dragonfly, neighbors: List[Dragonfly],
                             weights: Dict[str, float]) -> Dragonfly:
    """Update dragonfly position based on behavioral principles"""
    
    # Calculate behavioral vectors
    separation = calculate_separation(dragonfly, neighbors, weights['separation_radius'])
    alignment = calculate_alignment(dragonfly, neighbors)
    cohesion = calculate_cohesion(dragonfly, neighbors)
    food_attraction = calculate_food_attraction(dragonfly, food_source)
    enemy_distraction = calculate_enemy_distraction(dragonfly, enemy)
    
    # Update step vector
    new_step_vector = (weights['separation'] * separation +
                      weights['alignment'] * alignment +
                      weights['cohesion'] * cohesion +
                      weights['food'] * food_attraction +
                      weights['enemy'] * enemy_distraction)
    
    # Update position (simplified permutation update)
    new_position = dragonfly.position.copy()
    
    # Apply random swaps based on step vector magnitude
    step_magnitude = np.linalg.norm(new_step_vector)
    if step_magnitude > 0.1:
        # Perform random swaps to explore new permutations
        n_swaps = min(int(step_magnitude * 2), len(new_position) // 2)
        for _ in range(n_swaps):
            i, j = np.random.choice(len(new_position), 2, replace=False)
            new_position[i], new_position[j] = new_position[j], new_position[i]
    
    # Create updated dragonfly
    updated_dragonfly = Dragonfly(
        position=new_position,
        step_vector=new_step_vector,
        fitness=dragonfly.fitness,
        routes=dragonfly.routes
    )
    
    return updated_dragonfly

# Test position update
weights = {
    'separation': 0.1,
    'alignment': 0.1,
    'cohesion': 0.1,
    'food': 0.4,
    'enemy': 0.2,
    'separation_radius': 2.0
}

if len(swarm) > 0:
    updated = update_dragonfly_position(swarm[0], None, None, [], weights)
    print(f"Original position: {swarm[0].position}")
    print(f"Updated position: {updated.position}")

Original position: [2 4 5 6 3 1]
Updated position: [2 4 5 6 3 1]


In [ ]:
try:
    def dragonfly_algorithm(instance: EVRPInstance, distances: np.ndarray, 
                          energy_matrix: np.ndarray, population_size: int = 20, 
                          max_iterations: int = 100) -> Tuple[Dragonfly, List[float]]:
        """Dragonfly Algorithm for EVRP"""
        
        print("Running Dragonfly Algorithm for EVRP...")
        print(f"Population size: {population_size}, Max iterations: {max_iterations}")
        
        # Get number of customers
        customers = [i for i, node in enumerate(instance.nodes) if node.node_type == 'customer']
        n_customers = len(customers)
        
        # Initialize swarm
        swarm = initialize_dragonfly_swarm(population_size, n_customers)
        
        # Evaluate initial fitness
        for dragonfly in swarm:
            routes = decode_permutation_to_routes(dragonfly.position, instance, distances, energy_matrix)
            dragonfly.routes = routes
            dragonfly.fitness = calculate_route_fitness(routes, instance, distances, energy_matrix)
        
        # Initialize food source and enemy
        food_source = min(swarm, key=lambda d: d.fitness)
        enemy = max(swarm, key=lambda d: d.fitness)
        
        # Algorithm parameters
        weights = {
            'separation': 0.1,
            'alignment': 0.1,
            'cohesion': 0.1,
            'food': 0.4,
            'enemy': 0.2,
            'separation_radius': 3.0
        }
        
        # Track convergence
        convergence_history = [food_source.fitness]
        
        # Main optimization loop
        for iteration in range(max_iterations):
            # Update each dragonfly
            for i, dragonfly in enumerate(swarm):
                # Find neighbors (simplified as random subset)
                neighbor_indices = np.random.choice(len(swarm), min(5, len(swarm)-1), replace=False)
                neighbors = [swarm[j] for j in neighbor_indices if j != i]
                
                # Update position
                updated_dragonfly = update_dragonfly_position(
                    dragonfly, food_source, enemy, neighbors, weights
                )
                
                # Evaluate new fitness
                routes = decode_permutation_to_routes(updated_dragonfly.position, instance, distances, energy_matrix)
                updated_dragonfly.routes = routes
                updated_dragonfly.fitness = calculate_route_fitness(routes, instance, distances, energy_matrix)
                
                # Update if better
                if updated_dragonfly.fitness < dragonfly.fitness:
                    swarm[i] = updated_dragonfly
            
            # Update food source and enemy
            current_best = min(swarm, key=lambda d: d.fitness)
            current_worst = max(swarm, key=lambda d: d.fitness)
            
            if current_best.fitness < food_source.fitness:
                food_source = current_best
            
            if current_worst.fitness > enemy.fitness:
                enemy = current_worst
            
            # Track convergence
            convergence_history.append(food_source.fitness)
            
            # Print progress
            if (iteration + 1) % 10 == 0:
                print(f"Iteration {iteration + 1}: Best fitness = {food_source.fitness:.2f}")
        
        print(f"\nAlgorithm completed!")
        print(f"Best fitness: {food_source.fitness:.2f}")
        
        return food_source, convergence_history
    
    # Run the Dragonfly Algorithm
    start_time = time.time()
    best_dragonfly, convergence = dragonfly_algorithm(
        instance, distances, energy_consumption, 
        population_size=20, max_iterations=100
    )
    end_time = time.time()
    
    print(f"\nComputation Time: {end_time - start_time:.3f} seconds")
    print(f"Best solution fitness: {best_dragonfly.fitness:.2f}")
    print(f"Best routes: {best_dragonfly.routes}")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    def analyze_dragonfly_solution(best_dragonfly: Dragonfly, instance: EVRPInstance, 
                                  distances: np.ndarray, energy_matrix: np.ndarray) -> Dict:
        """Analyze the best solution found by the Dragonfly Algorithm"""
        
        analysis = {
            'best_fitness': best_dragonfly.fitness,
            'routes': best_dragonfly.routes,
            'num_routes': len(best_dragonfly.routes),
            'route_details': [],
            'total_distance': 0,
            'total_energy': 0,
            'customers_served': 0
        }
        
        customers = [i for i, node in enumerate(instance.nodes) if node.node_type == 'customer']
        served_customers = set()
        
        for i, route in enumerate(best_dragonfly.routes):
            route_distance = 0
            route_energy = 0
            route_customers = []
            route_charging = []
            
            for j in range(len(route) - 1):
                current = route[j]
                next_node = route[j + 1]
                
                route_distance += distances[current, next_node]
                route_energy += energy_matrix[current, next_node]
                
                if instance.nodes[next_node].node_type == 'customer':
                    route_customers.append(next_node)
                    served_customers.add(next_node)
                elif instance.nodes[next_node].node_type == 'charging_station':
                    route_charging.append(next_node)
            
            analysis['route_details'].append({
                'route_id': i,
                'nodes': route,
                'customers': route_customers,
                'charging_stops': route_charging,
                'distance': route_distance,
                'energy': route_energy,
                'battery_utilization': (route_energy / instance.vehicles[0].battery_capacity) * 100
            })
            
            analysis['total_distance'] += route_distance
            analysis['total_energy'] += route_energy
        
        analysis['customers_served'] = len(served_customers)
        
        return analysis
    
    # Analyze the solution
    solution_analysis = analyze_dragonfly_solution(best_dragonfly, instance, distances, energy_consumption)
    
    print("Dragonfly Algorithm Solution Analysis:")
    print("=" * 60)
    print(f"Total Distance: {solution_analysis['total_distance']:.2f} km")
    print(f"Total Energy Required: {solution_analysis['total_energy']:.2f} kWh")
    print(f"Number of Routes: {solution_analysis['num_routes']}")
    print(f"Customers Served: {solution_analysis['customers_served']}")
    
    print("\nRoute Details:")
    for detail in solution_analysis['route_details']:
        print(f"\nRoute {detail['route_id'] + 1}:")
        print(f"  Path: {' -> '.join([f'Node {n}' for n in detail['nodes']])}")
        print(f"  Customers: {detail['customers']}")
        print(f"  Charging Stops: {detail['charging_stops']}")
        print(f"  Distance: {detail['distance']:.2f} km")
        print(f"  Energy: {detail['energy']:.2f} kWh ({detail['battery_utilization']:.1f}% battery)")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'best_dragonfly' is not defined...]

In [ ]:
try:
    def visualize_dragonfly_solution(instance: EVRPInstance, best_dragonfly: Dragonfly, 
                                    analysis: Dict, convergence: List[float]):
        """Create comprehensive visualization of the Dragonfly Algorithm solution"""
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Electric Vehicle Routing Problem - Dragonfly Algorithm', fontsize=16, fontweight='bold')
        
        # Plot 1: Best solution routes
        colors = ['blue', 'red', 'green', 'orange', 'purple']
        
        # Plot nodes
        for i, node in enumerate(instance.nodes):
            if node.node_type == 'depot':
                ax1.scatter(node.x, node.y, s=300, c='black', marker='s', label='Depot', zorder=5)
                ax1.annotate('DEPOT', (node.x, node.y), xytext=(5, 5), textcoords='offset points', fontweight='bold')
            elif node.node_type == 'customer':
                ax1.scatter(node.x, node.y, s=200, c='blue', marker='o', label='Customer' if i == 1 else '', zorder=4)
                ax1.annotate(f'C{i}', (node.x, node.y), xytext=(5, 5), textcoords='offset points')
            else:  # charging station
                ax1.scatter(node.x, node.y, s=200, c='green', marker='^', label='Charging Station' if i == 7 else '', zorder=4)
                ax1.annotate(f'CS{i}', (node.x, node.y), xytext=(5, 5), textcoords='offset points')
        
        # Plot best routes
        for k, route in enumerate(best_dragonfly.routes):
            route_coords = [(instance.nodes[node].x, instance.nodes[node].y) for node in route]
            x_coords = [coord[0] for coord in route_coords]
            y_coords = [coord[1] for coord in route_coords]
            
            ax1.plot(x_coords, y_coords, color=colors[k % len(colors)], 
                    linewidth=2, alpha=0.7, marker='o', markersize=4,
                    label=f'Route {k+1}')
        
        ax1.set_xlabel('X Coordinate (km)')
        ax1.set_ylabel('Y Coordinate (km)')
        ax1.set_title('Best EV Routes Found by Dragonfly Algorithm')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Convergence curve
        ax2.plot(range(len(convergence)), convergence, linewidth=2, color='red', marker='o', markersize=3)
        ax2.set_xlabel('Iteration')
        ax2.set_ylabel('Best Fitness (Total Distance)')
        ax2.set_title('Dragonfly Algorithm Convergence')
        ax2.grid(True, alpha=0.3)
        
        # Add improvement annotation
        if len(convergence) > 1:
            improvement = ((convergence[0] - convergence[-1]) / convergence[0]) * 100
            ax2.annotate(f'Improvement: {improvement:.1f}%', 
                        xy=(len(convergence)-1, convergence[-1]), 
                        xytext=(10, 10), textcoords='offset points',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
        
        # Plot 3: Route performance comparison
        route_names = [f"Route {i+1}" for i in range(len(best_dragonfly.routes))]
        route_distances = [detail['distance'] for detail in analysis['route_details']]
        route_energies = [detail['energy'] for detail in analysis['route_details']]
        
        x = np.arange(len(route_names))
        width = 0.35
        
        bars1 = ax3.bar(x - width/2, route_distances, width, label='Distance (km)', alpha=0.7)
        ax3_twin = ax3.twinx()
        bars2 = ax3_twin.bar(x + width/2, route_energies, width, label='Energy (kWh)', alpha=0.7, color='orange')
        
        ax3.set_xlabel('Route')
        ax3.set_ylabel('Distance (km)')
        ax3_twin.set_ylabel('Energy (kWh)')
        ax3.set_title('Route Performance Analysis')
        ax3.set_xticks(x)
        ax3.set_xticklabels(route_names)
        ax3.legend(loc='upper left')
        ax3_twin.legend(loc='upper right')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Algorithm statistics and dragonfly behavior
        stats_text = f"""
        Dragonfly Algorithm Results:
        ─────────────────────
        Total Distance: {analysis['total_distance']:.2f} km
        Total Energy: {analysis['total_energy']:.2f} kWh
        Routes Used: {analysis['num_routes']}
        Customers Served: {analysis['customers_served']}
        Best Fitness: {best_dragonfly.fitness:.2f}
        
        Algorithm Parameters:
        ─────────────────────
        Population Size: 20 dragonflies
        Max Iterations: 100
        Convergence: {convergence[-1]:.2f} (final)
        Improvement: {((convergence[0] - convergence[-1]) / convergence[0]) * 100:.1f}%
        
        Dragonfly Behaviors:
        ─────────────────────
        • Separation: Collision avoidance
        • Alignment: Velocity matching
        • Cohesion: Center of mass attraction
        • Food Attraction: Best solution pursuit
        • Enemy Distraction: Worst solution avoidance
        
        Vehicle Specifications:
        ─────────────────────
        Battery Capacity: {instance.vehicles[0].battery_capacity} kWh
        Energy Rate: {instance.energy_consumption_rate} kWh/km
        """
        
        ax4.text(0.05, 0.5, stats_text, transform=ax4.transAxes, fontsize=9,
                 verticalalignment='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        ax4.set_xlim(0, 1)
        ax4.set_ylim(0, 1)
        ax4.axis('off')
        ax4.set_title('Algorithm Statistics', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    # Visualize the Dragonfly Algorithm solution
    visualize_dragonfly_solution(instance, best_dragonfly, solution_analysis, convergence)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'best_dragonfly' is not defined...]

In [ ]:
try:
    def compare_with_random_search(instance: EVRPInstance, distances: np.ndarray, 
                                  energy_matrix: np.ndarray, n_random: int = 100) -> Dict:
        """Compare Dragonfly Algorithm with random search"""
        
        print("Comparing with Random Search...")
        
        customers = [i for i, node in enumerate(instance.nodes) if node.node_type == 'customer']
        n_customers = len(customers)
        
        best_random_fitness = float('inf')
        best_random_routes = None
        
        # Random search
        for _ in range(n_random):
            random_permutation = np.random.permutation(customers)
            routes = decode_permutation_to_routes(random_permutation, instance, distances, energy_matrix)
            fitness = calculate_route_fitness(routes, instance, distances, energy_matrix)
            
            if fitness < best_random_fitness:
                best_random_fitness = fitness
                best_random_routes = routes
        
        # Analyze random solution
        random_analysis = analyze_dragonfly_solution(
            Dragonfly(position=np.array([]), step_vector=np.array([]), 
                     fitness=best_random_fitness, routes=best_random_routes),
            instance, distances, energy_matrix
        )
        
        # Compare results
        comparison = {
            'dragonfly_distance': solution_analysis['total_distance'],
            'random_distance': random_analysis['total_distance'],
            'improvement': ((random_analysis['total_distance'] - solution_analysis['total_distance']) / random_analysis['total_distance']) * 100,
            'dragonfly_routes': solution_analysis['num_routes'],
            'random_routes': random_analysis['num_routes'],
            'dragonfly_fitness': best_dragonfly.fitness,
            'random_fitness': best_random_fitness
        }
        
        print(f"\nComparison Results:")
        print(f"Dragonfly Algorithm: {comparison['dragonfly_distance']:.2f} km, {comparison['dragonfly_routes']} routes")
        print(f"Random Search: {comparison['random_distance']:.2f} km, {comparison['random_routes']} routes")
        print(f"Improvement: {comparison['improvement']:.1f}%")
        
        return comparison
    
    # Compare with random search
    comparison_results = compare_with_random_search(instance, distances, energy_consumption)
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    def test_robustness(instance: EVRPInstance, distances: np.ndarray, 
                      energy_matrix: np.ndarray, n_runs: int = 5) -> Dict:
        """Test algorithm robustness with multiple runs"""
        
        print(f"Testing Robustness with {n_runs} independent runs...")
        
        run_results = []
        
        for run in range(n_runs):
            print(f"Run {run + 1}/{n_runs}")
            
            # Set different random seed for each run
            np.random.seed(42 + run)
            random.seed(42 + run)
            
            # Run Dragonfly Algorithm
            best_dragonfly, convergence = dragonfly_algorithm(
                instance, distances, energy_matrix, 
                population_size=15, max_iterations=50  # Reduced for faster testing
            )
            
            analysis = analyze_dragonfly_solution(best_dragonfly, instance, distances, energy_matrix)
            
            run_results.append({
                'run': run + 1,
                'total_distance': analysis['total_distance'],
                'num_routes': analysis['num_routes'],
                'fitness': best_dragonfly.fitness,
                'convergence_iterations': len(convergence)
            })
        
        # Calculate statistics
        distances = [r['total_distance'] for r in run_results]
        fitness_values = [r['fitness'] for r in run_results]
        
        robustness_stats = {
            'mean_distance': np.mean(distances),
            'std_distance': np.std(distances),
            'min_distance': np.min(distances),
            'max_distance': np.max(distances),
            'mean_fitness': np.mean(fitness_values),
            'std_fitness': np.std(fitness_values),
            'coefficient_of_variation': (np.std(distances) / np.mean(distances)) * 100,
            'run_results': run_results
        }
        
        print(f"\nRobustness Analysis:")
        print(f"Mean Distance: {robustness_stats['mean_distance']:.2f} ± {robustness_stats['std_distance']:.2f} km")
        print(f"Range: [{robustness_stats['min_distance']:.2f}, {robustness_stats['max_distance']:.2f}] km")
        print(f"Coefficient of Variation: {robustness_stats['coefficient_of_variation']:.1f}%")
        
        return robustness_stats
    
    # Test robustness
    robustness_results = test_robustness(instance, distances, energy_consumption, n_runs=3)
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

### Why This Tier Exists vs Earlier Tiers

The Dragonfly Algorithm represents an advanced metaheuristic approach that overcomes limitations of both exact methods and simple heuristics:

**Advantages over Tier 1 (MIP):**
- **Scalability**: Handles larger instances (15+ customers) efficiently
- **Computational Speed**: Finds good solutions in seconds vs hours for MIP
- **Global Search**: Explores diverse solution regions to escape local optima
- **Adaptability**: Easy to modify for different constraints and objectives

**Advantages over Tier 2 (Savings Algorithm):**
- **Solution Quality**: Typically finds better solutions through global search
- **Exploration Capability**: Avoids greedy local optima traps
- **Population-Based Search**: Multiple search agents explore simultaneously
- **Balanced Exploration-Exploitation**: Five behavioral principles ensure good balance

**Disadvantages vs Earlier Tiers:**
- **No Optimality Guarantee**: Cannot prove solution optimality
- **Parameter Tuning**: Requires careful parameter selection
- **Stochastic Nature**: Different runs may produce different results
- **Computational Cost**: More expensive than simple heuristics

**When to Use This Tier:**
- Medium to large problems (15-50 customers) requiring high-quality solutions
- Problems where MIP is intractable but heuristics are insufficient
- Situations requiring balanced exploration and exploitation
- Multi-objective optimization problems
- Dynamic environments requiring robust solution methods

### Pros/Cons Summary

| Aspect | Pros | Cons |
|--------|------|------|
| Solution Quality | High-quality solutions, often near-optimal | No optimality guarantee |
| Scalability | Handles 20-50 customers effectively | Performance degrades for very large instances |
| Search Capability | Global search with local refinement | Requires parameter tuning |
| Robustness | Good performance across multiple runs | Stochastic nature creates variability |
| Complexity | Sophisticated search behavior | More complex implementation |

### Key Innovation: Swarm Intelligence for EVRP

The Dragonfly Algorithm brings several innovations to electric vehicle routing:

1. **Permutation Encoding**: Customer order encoded as dragonfly positions
2. **Behavioral Principles**: Five natural swarming behaviors guide search
3. **Adaptive Movement**: Step vectors control exploration vs exploitation
4. **Population Diversity**: Multiple agents maintain solution diversity
5. **Self-Organization**: Emergent intelligent behavior without central control

This tier demonstrates how nature-inspired algorithms can effectively solve complex logistics problems with electric vehicle constraints, providing a powerful alternative to traditional optimization methods.